# Porosity & Vshale Estimation from Volve Field Wells

**Author:** Ezeh Patrick Nkwachukwu  
**Dataset:** Equinor Volve Field Open Dataset — Wells **15/9-F-11B** and **15/9-F-12**  
**Tools:** Python · lasio · NumPy · Pandas · Matplotlib  

---

## Project Overview

In this notebook I performed  a petrophysical analysis of two wells from the publicly available **Equinor Volve Field dataset** (North Sea, Norway).  
The Volve field was operated by Equinor (formerly Statoil) and its data was released as an open dataset in 2018 for research and education.

### My Objectives
1. Load and visualize raw well log curves (GR, RHOB, NPHI, RT) for both wells
2. Compute **Vshale** from the Gamma Ray log
3. Estimate **density porosity (PHID)** using standard matrix-fluid equations
4. Identify and compare **reservoir intervals** across both wells using cutoffs



## 1. Import Libraries

In [ ]:
import lasio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings

warnings.filterwarnings('ignore')

# Set a clean plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 9

print("All libraries imported successfully.")

---
## 2. Load the LAS Files

We use **lasio** to read the LAS files. Each LAS file contains the well header metadata and the log curve data indexed by depth.

In [ ]:
# ── Load both wells ──────────────────────────────────────────────────────────
las_f11b = lasio.read("15_9-F-11B.las")
las_f12  = lasio.read("15_9-F-12.las")

print("Well 1 — 15/9-F-11B")
print(f"  Curves available: {[c.mnemonic for c in las_f11b.curves]}")
print(f"  Depth range: {las_f11b.index.min():.1f} – {las_f11b.index.max():.1f} m")

print("\nWell 2 — 15/9-F-12")
print(f"  Curves available: {[c.mnemonic for c in las_f12.curves]}")
print(f"  Depth range: {las_f12.index.min():.1f} – {las_f12.index.max():.1f} m")

---
## 3. Convert to Pandas DataFrames

I converted  to DataFrames because it makes it easier to work with the log data — filtering, computing new columns, and handling missing values.

In [ ]:
# ── Convert LAS to DataFrame ─────────────────────────────────────────────────
df_f11b = las_f11b.df().reset_index()
df_f12  = las_f12.df().reset_index()

# Rename the depth column for clarity
df_f11b.rename(columns={df_f11b.columns[0]: 'DEPTH'}, inplace=True)
df_f12.rename(columns={df_f12.columns[0]: 'DEPTH'}, inplace=True)

# Replace null/fill values with NaN
NULL_VAL = -9999.0
df_f11b.replace(NULL_VAL, np.nan, inplace=True)
df_f12.replace(NULL_VAL, np.nan, inplace=True)

print("Well 15/9-F-11B — first 3 rows:")
display(df_f11b.head(3))

print("\nWell 15/9-F-12 — first 3 rows:")
display(df_f12.head(3))

---
## 4. Identify Curve Name Variations

Different wells may use slightly different curve mnemonics (e.g. `GR` vs `GR_EDTC`).  
So I created a helper function finds the best available match for the curves we need.

In [ ]:
def find_curve(df, candidates):
    """
    This will return the first column name from `candidates` that exists in `df`.
    And returns None if none are found.
    """
    for name in candidates:
        if name in df.columns:
            return name
    return None

# Common mnemonic aliases for each curve
GR_NAMES   = ['GR', 'GR_EDTC', 'GR_RAW', 'GRD']
RHOB_NAMES = ['RHOB', 'RHOZ', 'DEN', 'RHOBC']
NPHI_NAMES = ['NPHI', 'NPHIL', 'NPOR', 'CALI_NPHI']
RT_NAMES   = ['RT', 'ILD', 'RDEP', 'RD', 'RDEEP', 'AT90', 'M2RX']

# ── Resolve curve names for each well ────────────────────────────────────────
curves_f11b = {
    'GR':   find_curve(df_f11b, GR_NAMES),
    'RHOB': find_curve(df_f11b, RHOB_NAMES),
    'NPHI': find_curve(df_f11b, NPHI_NAMES),
    'RT':   find_curve(df_f11b, RT_NAMES),
}

curves_f12 = {
    'GR':   find_curve(df_f12, GR_NAMES),
    'RHOB': find_curve(df_f12, RHOB_NAMES),
    'NPHI': find_curve(df_f12, NPHI_NAMES),
    'RT':   find_curve(df_f12, RT_NAMES),
}

print("Curves resolved for 15/9-F-11B:", curves_f11b)
print("Curves resolved for 15/9-F-12: ", curves_f12)

---
## 5. Visualize Raw Well Logs — Multi-Track Log Display

I plotted  four standard log tracks side-by-side for each well:
- **Track 1:** Gamma Ray (GR) — distinguishes shale from sand
- **Track 2:** Bulk Density (RHOB) — used for porosity estimation  
- **Track 3:** Neutron Porosity (NPHI) — sensitive to fluid content  
- **Track 4:** Resistivity (RT) — identifies hydrocarbons vs brine

In [ ]:
def plot_raw_logs(df, curves, well_name):
    """
    Plot a 4-track log display for a single well.
    """
    depth = df['DEPTH']

    fig, axes = plt.subplots(1, 4, figsize=(12, 14), sharey=True)
    fig.suptitle(f'Raw Well Logs — {well_name}', fontsize=13, fontweight='bold', y=1.01)

    track_cfg = [
        (curves['GR'],   'GR (API)',        'green',  (0,   200), False),
        (curves['RHOB'], 'RHOB (g/cc)',     'red',    (1.8, 3.0), False),
        (curves['NPHI'], 'NPHI (v/v)',      'blue',   (0.5, -0.05), False),  # reversed axis
        (curves['RT'],   'RT (ohm·m)',      'black',  (0.1, 1000), True),    # log scale
    ]

    for ax, (col, label, color, xlim, log_scale) in zip(axes, track_cfg):
        if col and col in df.columns:
            ax.plot(df[col], depth, color=color, linewidth=0.6)
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    transform=ax.transAxes, color='gray')

        ax.set_xlabel(label, fontsize=9)
        ax.set_xlim(xlim)
        if log_scale:
            ax.set_xscale('log')
        ax.invert_yaxis()
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.xaxis.set_label_position('top')
        ax.xaxis.tick_top()
        ax.tick_params(axis='x', labelsize=7)

    axes[0].set_ylabel('Depth (m)', fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{well_name.replace('/', '_')}_raw_logs.png", bbox_inches='tight')
    plt.show()
    print(f"Plot saved: {well_name.replace('/', '_')}_raw_logs.png")


# ── Plot both wells ───────────────────────────────────────────────────────────
plot_raw_logs(df_f11b, curves_f11b, '15/9-F-11B')
plot_raw_logs(df_f12,  curves_f12,  '15/9-F-12')

---
## 6. Compute Vshale from Gamma Ray Log

**Vshale** (volume of shale) is a key petrophysical parameter that tells us how "clean" a reservoir interval is.  
A Vshale of 0 = pure sand (clean reservoir); Vshale of 1 = pure shale.

### Formula — Linear GR Index:

$$V_{sh} = \frac{GR_{log} - GR_{min}}{GR_{max} - GR_{min}}$$

Where:
- **GR_min** = GR value in cleanest sand (typically 5th percentile of all GR readings)
- **GR_max** = GR value in pure shale (typically 95th percentile)

In [ ]:
def compute_vshale(df, gr_col):
    """
    Compute Vshale using the linear GR index method.
    GR_min and GR_max are estimated from the 5th and 95th percentiles
    of the GR log to avoid influence from extreme outliers.
    """
    gr = df[gr_col]
    gr_min = gr.quantile(0.05)   # clean sand value
    gr_max = gr.quantile(0.95)   # pure shale value

    vsh = (gr - gr_min) / (gr_max - gr_min)

    # Clamp to [0, 1] — values outside this range are non-physical
    vsh = vsh.clip(0, 1)

    print(f"  GR_min (clean sand) = {gr_min:.1f} API")
    print(f"  GR_max (pure shale) = {gr_max:.1f} API")
    return vsh, gr_min, gr_max


print("── Well 15/9-F-11B ──")
df_f11b['VSHALE'], gr_min_f11b, gr_max_f11b = compute_vshale(df_f11b, curves_f11b['GR'])

print("\n── Well 15/9-F-12 ──")
df_f12['VSHALE'],  gr_min_f12,  gr_max_f12  = compute_vshale(df_f12, curves_f12['GR'])

print("\nVshale computed for both wells.")
print(df_f11b[['DEPTH', curves_f11b['GR'], 'VSHALE']].dropna().head(5))

---
## 7. Estimate Density Porosity (PHID)

**Density porosity** is calculated from the bulk density log (RHOB) using the standard matrix-fluid equation:

$$\phi_D = \frac{\rho_{ma} - \rho_b}{\rho_{ma} - \rho_f}$$

Where:
- **ρ_ma** = matrix density = **2.65 g/cc** (sandstone/quartz — typical for Volve reservoir)
- **ρ_b** = bulk density from the RHOB log (g/cc)
- **ρ_f** = fluid density = **1.0 g/cc** (fresh water / brine assumption)

Result is clamped to a physically meaningful range of 0 to 0.45 (0–45% porosity).

In [ ]:
# ── Petrophysical constants ───────────────────────────────────────────────────
RHO_MA = 2.65   # Matrix density: sandstone/quartz (g/cc)
RHO_FL = 1.00   # Fluid density: brine (g/cc)


def compute_phid(df, rhob_col, rho_ma=RHO_MA, rho_fl=RHO_FL):
    """
    Compute density porosity from RHOB log.
    Formula: PHID = (rho_ma - RHOB) / (rho_ma - rho_fl)
    """
    rhob = df[rhob_col]
    phid = (rho_ma - rhob) / (rho_ma - rho_fl)

    # Clamp to physically valid range
    phid = phid.clip(0, 0.45)
    return phid


df_f11b['PHID'] = compute_phid(df_f11b, curves_f11b['RHOB'])
df_f12['PHID']  = compute_phid(df_f12,  curves_f12['RHOB'])

print("Density Porosity (PHID) computed for both wells.")
print("\nWell 15/9-F-11B — sample statistics:")
print(df_f11b['PHID'].dropna().describe().round(3))
print("\nWell 15/9-F-12 — sample statistics:")
print(df_f12['PHID'].dropna().describe().round(3))

---
## 8. Flag Reservoir Intervals Using Cutoffs

We define a **reservoir zone** as any depth interval that satisfies both:
- **Vshale < 0.40** (less than 40% shale — reasonably clean sand)
- **PHID > 0.10** (more than 10% porosity — porous enough to store fluids)

These are standard petrophysical cutoffs used as a first-pass screening.

In [ ]:
# ── Cutoff thresholds ─────────────────────────────────────────────────────────
VSH_CUTOFF  = 0.40   # Maximum Vshale for a reservoir interval
PHID_CUTOFF = 0.10   # Minimum density porosity for a reservoir interval


def flag_reservoir(df):
    """
    Flag depth intervals meeting both Vshale and PHID cutoffs as reservoir.
    Returns a boolean Series: True = reservoir, False = non-reservoir.
    """
    return (df['VSHALE'] < VSH_CUTOFF) & (df['PHID'] > PHID_CUTOFF)


df_f11b['RESERVOIR'] = flag_reservoir(df_f11b)
df_f12['RESERVOIR']  = flag_reservoir(df_f12)

# ── Summary ───────────────────────────────────────────────────────────────────
for name, df in [('15/9-F-11B', df_f11b), ('15/9-F-12', df_f12)]:
    total   = df['RESERVOIR'].count()
    res_cnt = df['RESERVOIR'].sum()
    pct     = 100 * res_cnt / total
    print(f"Well {name}: {res_cnt} / {total} depth samples flagged as reservoir ({pct:.1f}%)")

---
## 9. Petrophysical Summary Plot (Per Well)

This 5-track display shows the computed results alongside the raw logs:
- **Track 1:** GR with clean-sand / shale baselines  
- **Track 2:** RHOB  
- **Track 3:** NPHI  
- **Track 4:** Vshale (shaded green for clean reservoir)  
- **Track 5:** Density Porosity PHID (shaded blue for reservoir-quality porosity)

In [ ]:
def plot_petro_summary(df, curves, gr_min, gr_max, well_name):
    """
    5-track petrophysical summary plot for one well.
    """
    depth = df['DEPTH']
    res   = df['RESERVOIR']

    fig, axes = plt.subplots(1, 5, figsize=(15, 14), sharey=True)
    fig.suptitle(f'Petrophysical Summary — {well_name}', fontsize=13, fontweight='bold', y=1.01)

    # ── Track 1: GR ──
    ax = axes[0]
    if curves['GR'] and curves['GR'] in df.columns:
        ax.plot(df[curves['GR']], depth, color='green', linewidth=0.7)
        ax.axvline(gr_min, color='lime',   linestyle='--', linewidth=1, label=f'GR_min={gr_min:.0f}')
        ax.axvline(gr_max, color='darkgreen', linestyle='--', linewidth=1, label=f'GR_max={gr_max:.0f}')
        ax.legend(fontsize=7, loc='lower right')
    ax.set_xlabel('GR (API)', fontsize=9)
    ax.set_xlim(0, 200)

    # ── Track 2: RHOB ──
    ax = axes[1]
    if curves['RHOB'] and curves['RHOB'] in df.columns:
        ax.plot(df[curves['RHOB']], depth, color='red', linewidth=0.7)
    ax.set_xlabel('RHOB (g/cc)', fontsize=9)
    ax.set_xlim(1.8, 3.0)

    # ── Track 3: NPHI ──
    ax = axes[2]
    if curves['NPHI'] and curves['NPHI'] in df.columns:
        ax.plot(df[curves['NPHI']], depth, color='blue', linewidth=0.7)
    ax.set_xlabel('NPHI (v/v)', fontsize=9)
    ax.set_xlim(0.5, -0.05)  # Reversed — standard convention

    # ── Track 4: Vshale ──
    ax = axes[3]
    ax.plot(df['VSHALE'], depth, color='saddlebrown', linewidth=0.7)
    ax.axvline(VSH_CUTOFF, color='orange', linestyle='--', linewidth=1,
               label=f'Cutoff={VSH_CUTOFF}')
    # Shade clean reservoir zones
    ax.fill_betweenx(depth, 0, df['VSHALE'],
                     where=(df['VSHALE'] < VSH_CUTOFF),
                     color='lime', alpha=0.35, label='Clean sand')
    ax.set_xlabel('Vshale', fontsize=9)
    ax.set_xlim(0, 1)
    ax.legend(fontsize=7, loc='lower right')

    # ── Track 5: PHID ──
    ax = axes[4]
    ax.plot(df['PHID'], depth, color='navy', linewidth=0.7)
    ax.axvline(PHID_CUTOFF, color='cyan', linestyle='--', linewidth=1,
               label=f'Cutoff={PHID_CUTOFF}')
    # Shade high-porosity reservoir zones
    ax.fill_betweenx(depth, PHID_CUTOFF, df['PHID'],
                     where=(df['PHID'] > PHID_CUTOFF),
                     color='skyblue', alpha=0.45, label='Good porosity')
    ax.set_xlabel('PHID (v/v)', fontsize=9)
    ax.set_xlim(0, 0.45)
    ax.legend(fontsize=7, loc='lower right')

    # ── Common formatting ──
    for ax in axes:
        ax.invert_yaxis()
        ax.grid(True, linestyle='--', alpha=0.35)
        ax.xaxis.set_label_position('top')
        ax.xaxis.tick_top()
        ax.tick_params(axis='x', labelsize=7)

    axes[0].set_ylabel('Depth (m)', fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{well_name.replace('/', '_')}_petro_summary.png", bbox_inches='tight')
    plt.show()
    print(f"Plot saved: {well_name.replace('/', '_')}_petro_summary.png")


plot_petro_summary(df_f11b, curves_f11b, gr_min_f11b, gr_max_f11b, '15/9-F-11B')
plot_petro_summary(df_f12,  curves_f12,  gr_min_f12,  gr_max_f12,  '15/9-F-12')

---
## 10. Cross-Well Comparison — Reservoir Quality

Now we compare both wells side-by-side to identify **lateral variations in reservoir quality**.  
We overlay Vshale and PHID histograms for both wells, and plot a dual Vshale track.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Cross-Well Comparison: Vshale & PHID Distributions', fontsize=12, fontweight='bold')

# ── Vshale histogram ──
ax = axes[0]
ax.hist(df_f11b['VSHALE'].dropna(), bins=50, color='saddlebrown', alpha=0.6, label='15/9-F-11B')
ax.hist(df_f12['VSHALE'].dropna(),  bins=50, color='teal',        alpha=0.6, label='15/9-F-12')
ax.axvline(VSH_CUTOFF, color='red', linestyle='--', label=f'Cutoff = {VSH_CUTOFF}')
ax.set_xlabel('Vshale')
ax.set_ylabel('Frequency')
ax.set_title('Vshale Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# ── PHID histogram ──
ax = axes[1]
ax.hist(df_f11b['PHID'].dropna(), bins=50, color='navy',   alpha=0.6, label='15/9-F-11B')
ax.hist(df_f12['PHID'].dropna(),  bins=50, color='skyblue', alpha=0.7, label='15/9-F-12')
ax.axvline(PHID_CUTOFF, color='red', linestyle='--', label=f'Cutoff = {PHID_CUTOFF}')
ax.set_xlabel('PHID (v/v)')
ax.set_ylabel('Frequency')
ax.set_title('Density Porosity Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cross_well_histograms.png', bbox_inches='tight')
plt.show()
print("Plot saved: cross_well_histograms.png")

---
## 11. Petrophysical Summary Table

A concise summary of key statistics for both wells

In [ ]:
def summarize_well(df, well_name):
    """
    Build a summary statistics row for a well.
    """
    res_df = df[df['RESERVOIR'] == True]
    return {
        'Well':                   well_name,
        'Depth Range (m)':        f"{df['DEPTH'].min():.0f} – {df['DEPTH'].max():.0f}",
        'Mean Vshale':            round(df['VSHALE'].mean(), 3),
        'Mean PHID':              round(df['PHID'].mean(), 3),
        'Reservoir Intervals (%)': round(100 * df['RESERVOIR'].sum() / df['RESERVOIR'].count(), 1),
        'Avg PHID in Reservoir':  round(res_df['PHID'].mean(), 3) if len(res_df) > 0 else 'N/A',
        'Max PHID':               round(df['PHID'].max(), 3),
    }

summary_table = pd.DataFrame([
    summarize_well(df_f11b, '15/9-F-11B'),
    summarize_well(df_f12,  '15/9-F-12'),
])

summary_table = summary_table.set_index('Well')
print("\n=== PETROPHYSICAL SUMMARY TABLE ===")
display(summary_table)

---
### Key Observations
- Both wells penetrate intervals with good reservoir quality, but the proportion and depth of those intervals differ
- The cross-well comparison highlights **lateral variability** in reservoir character across the two well locations
- These results serve as a foundation for more advanced analysis (e.g., water saturation, net pay estimation)

---
